In [25]:
import json
import os
import re
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from datetime import date, timedelta

load_dotenv()

True

In [13]:
BASE_URL = "https://api.sectors.app/v2/"
OUTPUT_DIR = Path("output-sample")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = os.getenv("SECTORS_API_KEY")

if not API_KEY:
    raise RuntimeError("SECTORS_API_KEY tidak ditemukan di environment")

In [15]:
def call_request(endpoint, params=None, output_name=None, refresh=False):
    sample_name = output_name or endpoint.strip("/")
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", sample_name).strip("._")
    output_path = OUTPUT_DIR / f"{safe_name}.txt"

    if output_path.exists() and not refresh:
        data = json.loads(output_path.read_text(encoding="utf-8"))
        print(f"Menggunakan cache tanpa memakai kredit: {output_path}")
        return data

    url = BASE_URL + endpoint
    headers = {"Authorization": API_KEY}
    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30,
    )
    if not response.ok:
        print(f"API error {response.status_code}: {response.text}")
    response.raise_for_status()

    data = response.json()
    output_path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Output API tersimpan di: {output_path}")
    return data

## Broker

In [17]:
broker_activity_mg = call_request(
    "broker-activity/MG/",
    output_name="broker-activity_MG",
)

Menggunakan cache tanpa memakai kredit: output-sample\broker-activity_MG.txt


In [19]:
broker_activity_mg = call_request(
    "broker-summary/SINI/"
)

Output API tersimpan di: output-sample\broker-summary_SINI.txt


In [24]:
list_broker = call_request(
    "brokers/",
    params={"origin": "foreign"},
    output_name="broker-registry_foreign",
)

Output API tersimpan di: output-sample\broker-registry_foreign.txt


## Transaction Data

In [32]:
end_date = date.today().isoformat()
start_date = (date.today() - timedelta(days=60)).isoformat()


daily_SINI = call_request(
    "daily/SINI/",
    params={"start" : start_date, "end" : end_date},
    output_name="daily_SINI_60D"   
)

Output API tersimpan di: output-sample\daily_SINI_60D.txt


## Detailed Report

In [ ]:
# Company revenue segment

rev_ADMR = call_request(
    "company/get-segments/ADMR/",
    output_name="revenue_segments_ADMR"
)

Output API tersimpan di: output-sample\revenue_segments_ADMR.txt


In [38]:
# Shareholder composition

shareholder_SINI = call_request(
    "company/shareholders-composition/SINI/",
    output_name="shareholder_composition_SINI"
)

Output API tersimpan di: output-sample\shareholder_composition_SINI.txt


In [40]:
# Shareholder composition

shareholder_SINI = call_request(
    "free-float/",
    output_name="free_float"
)

Output API tersimpan di: output-sample\free_float.txt
